# Session 3: Protein Language Models

In this notebook, we're going to treat proteins as sequences first and biological molecules second. Each protein is a series of amino acids, which we can represent in text format, and then convert into tokens. The open-source model ESM2 gives us a way to turn those tokens into a numeric representation a smaller model can work with. We call this representation an **embedding**.

Full protein structure prediction is a computationally-expensive task, so instead, we're going to try something simpler: can we predict a biochemical property of our protein from the amino acid sequence alone?

Here, we're going to look at the reference proteome of *E. coli* K-12 and try to predict whether each protein is **soluble** or **insoluble** from its sequence embedding. The *ecoli_protein_properties.csv* file contains information about the physical and chemical properties of various proteins, including the **Soluble** property - a binary class assigned to each protein.

Working through this notebook, we're going to build a classifier which takes the amino acid sequence data as input and predicts whether the protein is soluble or insoluble. We'll encode our sequences using ESM2 and then train a simple MLP to classify them. The process of using the output features of a pre-trained model is called "feature extraction", which is a type of transfer learning (where we adapt a trained model to a different task).

Run the notebook one cell at a time and complete the activities to the best of your ability.

## 1. Setup

In this cell, we again load our libraries and fix a random seed. We're using mostly the same libraries, with the addition of:
- `pathlib` (`Path`) — A library which makes working with multiple file paths easier.
- `requests` — Allows us to download data from the internet.
- `pandas` (`pd`) — A package for working with tabular datasets. We'll use this to read in our E. coli protein properties CSV.
- `esm` — The package we use to download ESM-2.
- `Scikit-Learn` (`sklearn`) - We'll now also pull functions for re-scaling data.

In [ ]:
# These libraries are the same as before.
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, confusion_matrix, ConfusionMatrixDisplay,
)

# Helper library for working with file and folder paths
from pathlib import Path

# Lets us download files from the internet
import requests

# Tools for working with spreadsheet-like tables
import pandas as pd

# This is the important library - this contains the ESM2 model
import esm

# Again, we fix a random seed for each library
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set the device - if you have GPU, you should see cuda here (though our model is small enough to run on CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 2. Configuration

This is where we define our fixed **hyperparameters** - these are the variables which control how our model will behave. You can find an explanation of each parameter in the cell below. The notebook has reasonable defaults, so try running the full notebook once, then come back and change them.

In [ ]:
# Data paths - these are fixed unless you move your data around
# Folder where our data files live
DATA_DIR = Path('data_session3')
# Spreadsheet of E. coli protein properties (including the soluble/insoluble label)
ESOL_PATH = DATA_DIR / 'ecoli_protein_properties.csv'
# File containing the actual amino acid sequences for each protein
FASTA_PATH = DATA_DIR / 'ecoli_k12.fasta'
# Where we save the ESM2 embeddings to avoid recomputing them every run
EMB_CACHE = DATA_DIR / 'esm2_t12_35M_embeddings.npz'

# Classification threshold
# If the model's probability for "soluble" is above this number, we classify the protein as soluble
CLF_DECISION_THRESHOLD = 0.5

# ESM2 encoder properties - these are specific to the ESM2 model, no need to change these
# We're going to use the 35 million parameter model, which has 12 transformer layers.
ESM_MODEL_NAME = 'esm2_t12_35M_UR50D'
# The layer of ESM2 we want to read the protein features from (layer 12, the final layer)
ESM_LAST_LAYER = 12 
# The size of the embedding (how many numbers describe each protein after we run it through ESM2)
ESM_DIM = 480
# ESM2 has a max context window of 1024 tokens, including one start token, one end token, and 1022 amino acid tokens.
MAX_SEQ_LEN    = 1022

# Train-val-test split
# Fraction of proteins we'll use to train the classifier
TRAIN_FRAC = 0.70
# Fraction we'll use to check progress during training
VAL_FRAC   = 0.15
# Fraction we'll set aside for the final evaluation
TEST_FRAC  = 0.15
# Make sure these fractions actually sum to 100%
assert abs((TRAIN_FRAC + VAL_FRAC + TEST_FRAC) - 1.0) < 1e-9

# MLP hyperparameters
# How wide each hidden layer of the classifier is
HIDDEN_DIM = 32
# How many hidden layers the classifier has
NUM_LAYERS = 1
# Dropout rate
DROPOUT_RATE = 0.6

# Training hyperparameters
# How big a step the optimiser takes each update
LEARNING_RATE = 3e-4
# How many proteins the model sees in one go
BATCH_SIZE = 32
# Maximum number of full passes over the training set
NUM_EPOCHS = 30

## 3. Load the Protein Properties Dataset

In this cell, we're going to bring in the precomputed binary labels from `ecoli_protein_properties.csv`. Each row is one *E. coli* protein, and the `Soluble` column is defined as either `0` (insoluble) or `1` (soluble).

**Activity:** Use the `.head()` method to print the first few rows of the spreadsheet data.

In [ ]:
# Let's load the dataset and take a look
# Read the spreadsheet into a pandas table
df_eprop = pd.read_csv(ESOL_PATH)
# Now let's print the number of each class (seems pretty well-balanced)
n_pos = int(df_eprop['Soluble'].sum())
# Number of insoluble proteins (everything that wasn't labelled 1)
n_neg = len(df_eprop) - n_pos
# Print the total and per-class counts
print(f"Total proteins: {len(df_eprop)}  (insoluble: {n_neg}, soluble: {n_pos})")

## 4a. (Optional) Stream Dataset Directly from UniProt

You should find the data in the drive, but if you wanted to be clever and download it directly from a database like UniProt, you could use the following code.

Here, we set up a "request URL" for UniProt's REST API. This basically sends a message to the UniProt database, asking for the E. Coli K-12 data specifically, then UniProt streams the data back to us.

In [ ]:
# Define UniProt's reference proteome ID for E. coli K-12 MG1655.
PROTEOME_ID = 'UP000000625'
# Build the request URL
UNIPROT_URL = (
    'https://rest.uniprot.org/uniprotkb/stream?'
    f'format=fasta&query=proteome:{PROTEOME_ID}'
)

# Helper function to download the data
def download_uniprot_proteome(save_path: Path) -> None:
    # If we already downloaded the file, skip
    if save_path.exists():
        return
    # Ask UniProt for the file (give up after 120 seconds)
    response = requests.get(UNIPROT_URL, timeout=120)
    # Crash loudly if the download failed
    response.raise_for_status()
    # Save the downloaded text to disk
    save_path.write_text(response.text)

# Download the data
download_uniprot_proteome(FASTA_PATH)

## 4b. Load and Parse the Protein Sequences

The data comes in FASTA format: one header line, followed by the sequence itself. Fundamentally, this is text data.

We now need to match up the sequences in the FASTA file to the entries in our *ecoli_protein_properties.csv* file. To make matters even more fun, some genes are known by multiple names. The name of the gene might differ between our CSV and FASTA file, so we should check the "Synonyms of locus names K-12" column.

See if you can follow how the notebook matches gene names from the CSV to sequence entries from UniProt. If some entries fail to match, that is not unusual - real biological identifiers are rarely as tidy as we would like.


In [ ]:
# Define a helper function to parse the FASTA file
def parse_fasta(fasta_path: Path) -> dict[str, str]:
    # Define an empty dictionary to fill with gene name -> amino acid sequence mappings
    gene_sequences: dict[str, str] = {}
    current_gene = None
    current_seq: list[str] = []
    # Open the FASTA file
    with fasta_path.open() as fh:
        # Loop through every line in the file
        for line in fh:
            # Remove any leading/trailing whitespace
            line = line.strip()
            # Lines starting with ">" are gene headers
            if line.startswith('>'):
                # Save the previous sequence before starting a new one
                if current_gene and current_seq:
                    gene_sequences[current_gene] = ''.join(current_seq)
                # Reset trackers for the next entry
                current_gene = None
                current_seq = []
                # Look for the GN= token in the header which holds the gene name
                for part in line.split():
                    if part.startswith('GN='):
                        # Grab the gene name and stop scanning this header
                        current_gene = part.split('=', 1)[1]
                        break
            else:
                # Sequence lines just get appended to the current gene
                current_seq.append(line)
        # Don't forget the last gene at the end of the file
        if current_gene and current_seq:
            gene_sequences[current_gene] = ''.join(current_seq)
    # Hand back the finished dictionary
    return gene_sequences

# Run that function - this gives us a dictionary of gene names and corresponding sequences, e.g. {gene_name: sequence}
fasta_genes = parse_fasta(FASTA_PATH)
print(f"Parsed {len(fasta_genes)} sequences from FASTA")

# Now let's match up the gene names in the CSV to the sequences in the FASTA file
# Create an empty dictionary to map CSV gene name -> sequence
gene_to_sequence: dict[str, str] = {}
# Create a list to store gene names that we couldn't find anywhere in the FASTA file
unmatched: list[str] = []
# Loop over every row in the CSV
for _, row in df_eprop.iterrows():
    # Get the official gene name in the CSV
    name = row['Gene name K-12']
    # Check if the name appears directly in the FASTA dictionary
    if name in fasta_genes:
        gene_to_sequence[name] = fasta_genes[name]
        continue
    # The synonyms column is optional, so we check if it exists
    synonyms = row.get('Synonyms of locus names K-12', '')
    # Set up flag for whether any synonym worked
    matched = False
    # If the synonyms cell isn't empty, try each one
    if pd.notna(synonyms):
        # Synonyms are pipe-separated, so we separate them by "|"
        for s in str(synonyms).split('|'):
            # Clean up any whitespace around the synonym
            s = s.strip()
            # Use the first synonym that has a matching FASTA entry
            if s and s in fasta_genes:
                gene_to_sequence[name] = fasta_genes[s]
                matched = True
                break
    # If nothing matched, remember this gene for later inspection
    if not matched:
        unmatched.append(name)


# Make a note of how many entries we had before filtering
n_before = len(df_eprop)
# Filter out the rows that don't have a matching sequence
df_eprop = df_eprop[df_eprop['Gene name K-12'].isin(gene_to_sequence.keys())].reset_index(drop=True)
# We lose a few hundred, but we still have more than enough
print(f"Rows in df_eprop: {n_before} before filtering -> {len(df_eprop)} after filtering")

## Max token length
ESM2, like most models, has a "max token length". In this case, the max token length is 1022, so we can either remove amino sequences with more than 1022 amino acids, or truncate them. Since we don't have much data, we'll opt to truncate them here. Let's plot the sequence length distribution as a histogram to see how many proteins we'll need to cut.

In [ ]:
# Define an array containing the length of every protein in the dataset
seq_lengths = np.array([len(gene_to_sequence[g]) for g in df_eprop['Gene name K-12']])
# Record labels (soluble or not) as a plain numpy array
y_tmp = df_eprop['Soluble'].to_numpy()

# Set up a figure with two side-by-side panels
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Define equally-spaced bin edges for the histogram
bins = np.linspace(0, max(int(seq_lengths.max()), MAX_SEQ_LEN), 60)
# Create histogram of insoluble proteins by length
axes[0].hist(seq_lengths[y_tmp == 0], bins=bins, color='steelblue',  alpha=0.6, label='insoluble')
# Create histogram of soluble proteins by length, on the same axes
axes[0].hist(seq_lengths[y_tmp == 1], bins=bins, color='darkorange', alpha=0.6, label='soluble')
# Vertical line marking ESM2's maximum length
axes[0].axvline(MAX_SEQ_LEN, color='red', linestyle='--', linewidth=0.8, label=f'ESM2 context limit ({MAX_SEQ_LEN})')
# Label the x-axis
axes[0].set_xlabel('sequence length (residues)')
# Label the y-axis
axes[0].set_ylabel('number of proteins')
# Set the panel title
axes[0].set_title('Sequence length by class')
# Show the per-class legend
axes[0].legend()
# The red line marks where ESM2 starts truncating - if any sequences sit beyond it, we have to truncate them

# While we're at it, let's see if we can find any obvious link between amino acid composition and solubility.
# We'll also plot the average amino acid composition by class.
# Here, we define the 20 standard amino acid one-letter codes
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')
# Define a helper function to count how often each amino acid shows up across a list of sequences
def aa_frequencies(seqs: list[str]) -> np.ndarray:
    # Initialise per-amino-acid counter
    counts = {aa: 0 for aa in AMINO_ACIDS}
    # Total residues counted so far
    total = 0
    # Loop over every sequence
    for s in seqs:
        # And every residue in that sequence
        for ch in s:
            # Skip anything that isn't one of the 20 standard residues
            if ch in counts:
                counts[ch] += 1
                total += 1
    # Convert raw counts into per-amino-acid frequencies
    return np.array([counts[aa] / max(total, 1) for aa in AMINO_ACIDS])

# List of sequences belonging to the insoluble class
seqs_neg = [gene_to_sequence[g] for g, v in zip(df_eprop['Gene name K-12'], y_tmp) if v == 0]
# List of sequences belonging to the soluble class
seqs_pos = [gene_to_sequence[g] for g, v in zip(df_eprop['Gene name K-12'], y_tmp) if v == 1]
# Amino acid frequencies for insoluble proteins
freq_neg = aa_frequencies(seqs_neg)
# Amino acid frequencies for soluble proteins
freq_pos = aa_frequencies(seqs_pos)

# Let's create a bar chart
# Define the x-axis positions for the 20 amino acid bars
x = np.arange(len(AMINO_ACIDS))
# Width of each side-by-side bar
width = 0.4
# Bars for insoluble proteins (left)
axes[1].bar(x - width / 2, freq_neg, width, color='steelblue',  label='insoluble')
# Bars for soluble proteins (right)
axes[1].bar(x + width / 2, freq_pos, width, color='darkorange', label='soluble')
# Place ticks on each amino acid position
axes[1].set_xticks(x)
# Label the ticks with the amino acid letters
axes[1].set_xticklabels(AMINO_ACIDS)
# Label the x-axis
axes[1].set_xlabel('amino acid')
# Label the y-axis
axes[1].set_ylabel('frequency')
# Set the panel title
axes[1].set_title('Amino acid composition by class')
# Show legend
axes[1].legend()

# Tidy subplot spacing
plt.tight_layout()
# Show the figure
plt.show()

# Looks pretty evenly-matched, it would be difficult to estimate solubility from sequence length and composition alone.

## 5. Encode Sequences with ESM2

Here we ask a pre-trained protein language model to read each amino-acid sequence and turn it into an embedding vector. This version of the model outputs a vector of length 480, but this differs per model. First we tokenise our sequence. In this case, it's very simple - we just assign an integer to each different amino acid.

Remember that ESM2 is a pre-trained model, so we're not going to retrain it from scratch. We're just borrowing the representation it already learned from many protein sequences.

In this case, each residue/amino acid gets its own embedding vector, and then we average across the sequence to get one 480-number summary per protein.

In [ ]:
# Quick test - let's see what the tokens and embedding look like for one example sequence
# Grab the sequence of one specific gene
test_seq = gene_to_sequence['yaaX']
# Here, alphabet is ESM's built-in tokeniser - it converts amino acids to numbers
model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
# Let's put the model in evaluation mode, since we aren't training
model = model.to(device).eval()
# Get our tokens out: get_batch_converter() returns a callable, which we then call with the batch
batch_converter = alphabet.get_batch_converter()
# Convert our test sequence into a tensor of numeric tokens (we include the string "test" here because batch_converter expects a label)
_, _, tokens = batch_converter([('test', test_seq)])
print(f"Shape of token vector: {tokens.shape}")
print(f"Tokens: {tokens}")
# Print the tokens - there are 98 residues in this sequence, so we get 98 tokens + 2 for start and end tokens

# Now let's get the embedding by passing our tokens to the model
# Turn off autograd
with torch.no_grad():
    # Run ESM2 and ask it to return the activations from the final layer
    out = model(tokens.to(device), repr_layers=[ESM_LAST_LAYER])
# Slice out the per-residue features, ignoring the start/end tokens
per_residue = out['representations'][ESM_LAST_LAYER][0, 1:len(test_seq)+1]
# Print the shape of the embedding - 98 residues, 480 dimensions
print(f"Shape of embedding vector: {per_residue.shape}")
# Print the first residue's embedding - total nonsense to you and me, but very informative to a computer
print(f"First residue's embedding: {per_residue[0]}")
# Look at that - we've managed to convert our amino acid sequence into numbers. Yay

## Encode all sequences
Now that we've seen what tokenisation and embedding look like, we can encode all of our amino acid sequences.

In [ ]:
# We'll define a helper function to encode our sequences
def encode_sequences(gene_names: list[str], sequences: list[str]) -> np.ndarray:
    # Set up the model and tokeniser
    model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()
    # Move the model to device and evaluation mode
    model = model.to(device).eval()
    # Define an empty array to store our embeddings
    embs = np.zeros((len(sequences), ESM_DIM), dtype=np.float32)
    # Re-use the same tokeniser as before
    bc = alphabet.get_batch_converter()
    # Since our max context window is 1022, we may have to truncate some amino acids - we'll track those here
    n_truncated = 0
    # Loop through our sequences
    for i, seq in enumerate(sequences):
        # Check if we have to truncate
        seq_t = seq[:MAX_SEQ_LEN]
        # If we did chop the sequence, count it
        if len(seq) > MAX_SEQ_LEN:
            n_truncated += 1
        # Tokenise and embed
        _, _, batch_tokens = bc([(gene_names[i], seq_t)])
        # Move the tokens to the chosen device
        batch_tokens = batch_tokens.to(device)
        # Disable autograd
        with torch.no_grad():
            # Run ESM2 and ask for the final layer's activations
            out = model(batch_tokens, repr_layers=[ESM_LAST_LAYER])
            # Take the mean over all residues to get one vector per protein
            emb = out['representations'][ESM_LAST_LAYER][0, 1:len(seq_t) + 1].mean(0)
        # Store this protein's vector in the embeddings array
        embs[i] = emb.cpu().numpy()
    # If any sequences were too long, mention how many
    if n_truncated:
        print(f"  truncated {n_truncated} sequences longer than {MAX_SEQ_LEN} residues")
    # Return the matrix of embeddings (one row per protein)
    return embs

# Prepare our aligned gene names and sequences
gene_names_aligned = df_eprop['Gene name K-12'].tolist()
# Sequences in matching order so row i lines up with name i
sequences_aligned  = [gene_to_sequence[g] for g in gene_names_aligned]

# Start with no embeddings - we'll fill this in
X_esm = None
# Check whether we've already saved embeddings from a previous run
if EMB_CACHE.exists():
    # Load the saved embeddings and the gene names they belong to
    cached = np.load(EMB_CACHE, allow_pickle=True)
    # Map each cached gene name to its row index
    name_to_idx = {n: i for i, n in enumerate(cached['gene_names'])}
    # Only use the cache if every gene we need is in it
    if all(g in name_to_idx for g in gene_names_aligned):
        # Look up the cached row for each gene we need
        idx = np.array([name_to_idx[g] for g in gene_names_aligned])
        # Pull out the right rows in the right order
        X_esm = cached['embeddings'][idx].astype(np.float32)

# If we couldn't use the cache, compute embeddings now
if X_esm is None:
    # Run every sequence through ESM2
    X_esm = encode_sequences(gene_names_aligned, sequences_aligned)
    # Save the freshly computed embeddings so the next run is fast
    np.savez(EMB_CACHE, embeddings=X_esm, gene_names=np.array(gene_names_aligned))

# 1850 sequences, each with an embedding of length 480
print(f"Embedding matrix shape: {X_esm.shape}")


## Classical dimensionality reduction
Could this be enough to classify the proteins? 480 dimensions is a lot, maybe a bit of classic dimensionality reduction will tell us more. We'll use t-SNE to reduce the dimensionality to 2D and plot each sequence. We won't go into the mathematics here, all we need to know is that t-SNE will try to find a way of compressing our data into two dimensions while conserving local neighbourhoods (i.e. proteins which are close together in 480-dimensional space should also be close in 2-dimensional space).

In [ ]:
# Import t-SNE - a tool that squashes high-dimensional data down to 2D for plotting
from sklearn.manifold import TSNE

# Standardise once (this is for visualisation only - it doesn't affect the model)
X_vis = StandardScaler().fit_transform(X_esm)

# t-SNE preserves local neighbourhoods, so similar proteins should end up near each other
tsne = TSNE(n_components=2, perplexity=30, init='pca', random_state=SEED)
# Run t-SNE
X_tsne = tsne.fit_transform(X_vis)

# Get solubility labels
labels_vis = df_eprop['Soluble'].to_numpy()
# Create the figure that will hold our scatter plot
fig, ax = plt.subplots(figsize=(7, 6))
# Plot each class as its own scatter plot
for cls, color, name in [(0, 'steelblue', 'insoluble'), (1, 'darkorange', 'soluble')]:
    # Boolean mask selecting proteins belonging to each class
    mask = labels_vis == cls
    # Plot proteins
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               s=8, alpha=0.5, color=color, label=name)
# Add the title
ax.set_title('t-SNE of ESM2 embeddings (perplexity 30)')
# Label the x-axis
ax.set_xlabel('component 1')
# Label the y-axis
ax.set_ylabel('component 2')
# Show the class legend
ax.legend()

# Tidy spacing
plt.tight_layout()
# Show the figure
plt.show()

# Now, ESM2 was never told about solubility, it was actually trained to predict missing amino acids in sequence context.
# However, in our case, we still see some clustering of soluble and insoluble proteins in the embedding space, even after reducing to 2D.
# Somehow, by encoding the protein sequence, we also encode information about solubility.

## 6. Build targets

Now we have our embeddings, let's get ready to train a simple MLP to classify them. We'll start by gathering our input data.

In [ ]:
# Get our target labels and embeddings
# Labels (0 = insoluble, 1 = soluble) as a NumPy array
y = df_eprop['Soluble'].to_numpy(dtype=np.float32)
# Inputs - one ESM2 embedding per protein
X = X_esm

# Check the distribution of our target labels - still seems balanced!
# Count of soluble proteins
n_pos = int(y.sum())
# Count of insoluble proteins
n_neg = int(len(y) - n_pos)
# Total number of proteins after filtering
print(f"Classification: n = {len(y)}")
print(f"  insoluble (label 0): {n_neg}")
print(f"  soluble   (label 1): {n_pos}")
print(f"  positive (soluble) fraction: {y.mean():.3f}")


## 7. Train-Validation-Test Split

Now we perform our train-val-test split, just as we did before.

**Disclaimer:** In a real protein ML project, you would want to be stricter - similar proteins should be grouped so that close relatives do not appear in both the train and test sets, otherwise we risk inadvertently memorising the features of the test data. For the purpose of demonstration, we just use a random split.

In [ ]:
# Shuffle and split the data into train, val, and test sets
# Get a random ordering of all the protein indices
rng = np.random.default_rng(SEED)
perm = rng.permutation(len(X))
# Define how many proteins go into the training set
n_train = int(round(TRAIN_FRAC * len(X)))
# Define how many proteins go into the validation set
n_val   = int(round(VAL_FRAC   * len(X)))
# Split the shuffled list into the three groups
train_idx, val_idx, test_idx = np.split(perm, [n_train, n_train + n_val])
# Pull out the input features for each split
X_train, X_val, X_test = X[train_idx], X[val_idx], X[test_idx]
# Pull out the matching labels for each split
y_train, y_val, y_test = y[train_idx], y[val_idx], y[test_idx]

# Standardise features using train-set statistics only (this improves training stability)
scaler = StandardScaler().fit(X_train)
# Apply the scaler to the training, validation and test features
X_train_s = scaler.transform(X_train).astype(np.float32)
X_val_s = scaler.transform(X_val).astype(np.float32)
X_test_s = scaler.transform(X_test).astype(np.float32)

# Let's see what we end up with
# Report the actual sizes of each split
print(f"Split sizes (target {TRAIN_FRAC:.0%}/{VAL_FRAC:.0%}/{TEST_FRAC:.0%}): "
      f"train={len(X_train)}, val={len(X_val)}, test={len(X_test)}")
# And the soluble proportion in each, so we know they're roughly balanced
print(f"Soluble fraction - train: {y_train.mean():.3f}, "
      f"val: {y_val.mean():.3f}, test: {y_test.mean():.3f}")

## 8. Datasets and DataLoaders

In this cell, we're going to package the arrays into the tensor format PyTorch expects and set up another `DataLoader` so we can feed the model batches instead of the whole dataset at once.

In [ ]:
# Here we set up a custom dataset class that PyTorch can use
class ProteinSolubilityDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray):
        # Store the encoded features as a PyTorch tensor
        self.X = torch.from_numpy(X.astype(np.float32))
        # Store the labels as a PyTorch tensor
        self.y = torch.from_numpy(y.astype(np.float32))

    # Tell PyTorch how many items are in the dataset
    def __len__(self) -> int:
        return len(self.X)

    # Tell PyTorch how to fetch one (features, label) pair by index
    def __getitem__(self, idx: int) -> tuple[torch.Tensor, torch.Tensor]:
        return self.X[idx], self.y[idx]

# Define our data loaders
# Training loader - shuffled so each epoch sees data in a different order
train_loader = DataLoader(ProteinSolubilityDataset(X_train_s, y_train),
                          batch_size=BATCH_SIZE, shuffle=True)
# Validation loader - fixed order (no need to shuffle)
val_loader = DataLoader(ProteinSolubilityDataset(X_val_s,   y_val),
                          batch_size=BATCH_SIZE, shuffle=False)
# Test loader - also fixed order
test_loader = DataLoader(ProteinSolubilityDataset(X_test_s,  y_test),
                          batch_size=BATCH_SIZE, shuffle=False)

## 9. MLP Classifier

Here we add a small neural network on top of the ESM2 embedding. ESM2 has already done the hard work of reading the protein sequence - this classifier only has to decide whether the resulting vector looks more soluble or insoluble.

**Activity:** Try changing the number of hidden layers and hidden dimension. But be warned, with a small dataset, a bigger model may simply memorise patterns in the split. If you do increase the width or number of layers, watch what happens to the validation loss.

In [ ]:
# Define our simple MLP classifier
class SolubilityPredictor(nn.Module):
    # Input important hyperparameters (defined in the Configuration cell)
    def __init__(self, input_dim: int, hidden_dim: int = HIDDEN_DIM,
                 num_layers: int = NUM_LAYERS, dropout: float = DROPOUT_RATE):
        super().__init__()
        # Define an empty list we'll fill with layers
        layers: list[nn.Module] = []
        # Track the current input size as we add layers
        in_d = input_dim
        # Here is where we define our layers, based on the configuration hyperparameters we set earlier
        for _ in range(num_layers):
            # Each block is a linear layer, with ReLU activation and dropout
            layers += [nn.Linear(in_d, hidden_dim), nn.ReLU(), nn.Dropout(dropout)]
            # The next layer's input width is the previous hidden width
            in_d = hidden_dim
        # Final output layer: one number per protein (this will eventually become the probability of being soluble)
        layers.append(nn.Linear(in_d, 1))
        # Wrap all the layers into a single block
        self.net = nn.Sequential(*layers)

    # How the model turns an input into a prediction
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Run through the network and drop the unneeded final dimension
        return self.net(x).squeeze(-1)

# Define our model
# Input dimension equals the embedding size (480)
input_dim = X_train_s.shape[1]
# Build the model and move it onto the chosen device
model = SolubilityPredictor(input_dim).to(device)
print(model)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

## 10. Training Loop

Now we let the classifier learn. This is similar to the MLP we trained in the last notebook, so we'll use the same optimiser (Adam) and loss (cross entropy).

In [ ]:
# Define our loss and optimiser
criterion = nn.BCEWithLogitsLoss()
optimiser = optim.AdamW(model.parameters(), lr=LEARNING_RATE)

# Define a helper function to run one epoch and return (mean_loss, accuracy) - this can be used for training or validation
def run_epoch(model, loader, criterion, optimiser=None) -> tuple[float, float]:
    # Set model to train (when an optimiser is given)
    is_train = optimiser is not None
    model.train(is_train)
    # Initialise variables to track loss and accuracy
    total_loss = 0.0
    # Count of correct predictions
    n_correct = 0
    # Count of total predictions seen so far
    n = 0
    # Loop through the data loader
    for xb, yb in loader:
        # Get embedding and class
        xb, yb = xb.to(device), yb.to(device)
        # Zero gradients
        if is_train:
            optimiser.zero_grad()
        # Forward pass
        with torch.set_grad_enabled(is_train):
            # Run the model to get raw scores (logits)
            logits = model(xb)
            # Compare the scores against the true labels
            loss = criterion(logits, yb)
        # Backward pass and optimisation
        if is_train:
            # Compute gradients for every parameter
            loss.backward()
            # Apply one optimisation step
            optimiser.step()
        # Record loss and accuracy
        total_loss += float(loss.item()) * len(xb)
        # Convert scores to predictions using the threshold
        preds = (torch.sigmoid(logits.detach()) >= CLF_DECISION_THRESHOLD).float()
        # Count how many predictions match the true labels
        n_correct += int((preds == yb).sum().item())
        # Track total proteins processed
        n += len(xb)
    # Return mean loss and overall accuracy for the epoch
    return total_loss / max(n, 1), n_correct / max(n, 1)

# Initialise lists to store our training and validation loss and accuracy
train_losses = []
val_losses = []
train_accs = []
val_accs = []

# Main training loop
for epoch in range(1, NUM_EPOCHS + 1):
    # Train one epoch
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimiser)
    # Validate one epoch
    val_loss, val_acc = run_epoch(model, val_loader,   criterion)
    # Store the loss and accuracy
    train_losses.append(train_loss)
    # Store the validation loss
    val_losses.append(val_loss)
    # Store the training accuracy
    train_accs.append(train_acc)
    # Store the validation accuracy
    val_accs.append(val_acc)

    # Print the loss and accuracy at every 10th epoch and the first epoch
    if epoch % 10 == 0 or epoch == 1:
        # Print a progress line for this epoch
        print(f"Epoch {epoch:>3d}/{NUM_EPOCHS} "
              f"train_loss = {train_loss:.4f} val_loss = {val_loss:.4f} "
              f"train_acc = {train_acc:.3f} val_acc = {val_acc:.3f}")

# Final-epoch validation results (we use the model as it stands after the last epoch)
print(f"\nFinal validation loss: {val_losses[-1]:.4f}")
print(f"Final validation accuracy: {val_accs[-1]:.3f}")

## Plot training curves

In [ ]:
# Set x axis
epochs_axis = np.arange(1, len(train_losses) + 1)

# Set up two side-by-side panels
fig, axes = plt.subplots(1, 2, figsize=(16, 4.2))

# Grab the first panel
ax = axes[0]
# Plot training loss against epoch
ax.plot(epochs_axis, train_losses, label='train', color='steelblue')
# Plot validation loss against epoch
ax.plot(epochs_axis, val_losses,   label='val',   color='darkorange')
# Label the x-axis
ax.set_xlabel('epoch')
# Label the y-axis
ax.set_ylabel('BCEWithLogitsLoss')
# Set the panel title
ax.set_title('Training curves')
# Show the legend
ax.legend()

# Grab the second panel
ax = axes[1]
# Plot training accuracy per epoch
ax.plot(epochs_axis, train_accs, label='train', color='steelblue')
# Plot validation accuracy per epoch
ax.plot(epochs_axis, val_accs,   label='val',   color='darkorange')
# Label the x-axis
ax.set_xlabel('epoch')
# Label the y-axis
ax.set_ylabel('accuracy')
# Force the y-axis to span the full 0-1 range
ax.set_ylim(0.0, 1.0)
# Set the panel title
ax.set_title('Per-epoch accuracy')
# Show the legend
ax.legend()

# Tidy spacing between panels
plt.tight_layout()
# Show the figure
plt.show()

## 11. Evaluation

Now it's time to evaluate our model on the held-out test set. We'll calculate a few plots and statistics, including:
- A probability histogram - This shows the histogram of probabilities (of being soluble) we predict for each amino acid sequence.
- A confusion matrix (as in the previous notebook).

**Activity:** Try changing the `CLF_DECISION_THRESHOLD`. What happens if it becomes too low or too high?

In [ ]:
# Extract logits and probabilities for the test set
# Define a helper function to pull out logits
@torch.no_grad()
def predict_logits(model, X: np.ndarray) -> np.ndarray:
    # Switch the model to evaluation mode
    model.eval()
    # Convert the input array to a tensor on the right device
    xb = torch.from_numpy(X.astype(np.float32)).to(device)
    # Run the model and return predictions as a numpy array
    return model(xb).cpu().numpy()

# Get raw model logits on the test set
test_logits = predict_logits(model, X_test_s)
# Turn logits into probabilities by applying a sigmoid transformation
test_proba = 1.0 / (1.0 + np.exp(-test_logits))
# Convert probabilities to 0/1 predictions using our decision threshold
test_pred = (test_proba >= CLF_DECISION_THRESHOLD).astype(int)
# True labels as integers (for the scoring functions)
y_test_int = y_test.astype(int)

# Calculate accuracy
acc = float(accuracy_score(y_test_int, test_pred))
# Get confusion matrix
cm = confusion_matrix(y_test_int, test_pred, labels=[0, 1])
print(f"Accuracy: {acc:.3f}")

# Plot our probability histogram and confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))

# Probability histogram - proteins on the right of the red line are classified as soluble, while those on the left
# are classified as insoluble. We can see a very clear separation between the two classes, which is good news

# Set equally-spaced bins from 0 to 1
bins = np.linspace(0, 1, 25)
# Create histogram of predicted probabilities for proteins that are actually insoluble
axes[0].hist(test_proba[y_test_int == 0], bins=bins, color='steelblue',
             alpha=0.6, label='true: insoluble')
# Create histogram of predicted probabilities for proteins that are actually soluble
axes[0].hist(test_proba[y_test_int == 1], bins=bins, color='darkorange',
             alpha=0.6, label='true: soluble')
# Add vertical line at the decision threshold
axes[0].axvline(CLF_DECISION_THRESHOLD, color='red', linestyle='--', linewidth=0.8,
                label=f'threshold = {CLF_DECISION_THRESHOLD}')
# Label the x-axis
axes[0].set_xlabel('predicted probability of soluble')
# Label the y-axis
axes[0].set_ylabel('number of proteins')
# Set the panel title
axes[0].set_title('Predicted probability by true class')
# Show the legend
axes[0].legend()

# Confusion matrix - again, this shows us how many proteins were correctly (and incorrectly) assigned to each class
ConfusionMatrixDisplay(cm, display_labels=['insoluble', 'soluble']).plot(
    ax=axes[1], cmap='Blues', colorbar=False, values_format='d',
)
# Title showing the overall accuracy
axes[1].set_title(f'Confusion matrix (acc = {acc:.3f})')

# Tidy spacing
plt.tight_layout()
# Show the figure
plt.show()


## 12. Where Could You Take This Next?

This is a working mock answer, not the final form of a serious solubility predictor. If you wanted to take this forward, you could try one or more of the following:

### A. Get more labelled proteins
The dataset is quite small for a 480-dimensional embedding. You could look for extra labels in resources such as DeepSol, SoluProt, or structural genomics datasets.

### B. Pool the sequence differently
Mean pooling is easy, but it treats every position as equally important. We could instead concatenate mean/max/min pooling, learn an attention-pooling layer, or pool different sequence regions separately.

### C. Fine-tune part of ESM2
Right now, ESM2 is frozen and only the small classifier learns. With more data and more compute, you could unfreeze the last few transformer layers and let the representation adapt to solubility.

### D. Try a larger protein language model
The ESM2 family includes much larger models than the one used here. Larger embeddings may carry richer sequence information, though they will cost more time and memory.

### E. Change the prediction head
The MLP is intentionally plain. You could try a small transformer over residue-level embeddings, a 1D convolutional model for local motifs, or a multi-task head that predicts solubility alongside related properties.

### F. Make the evaluation stricter
For real protein ML, replace the random split with a sequence-grouped split using tools such as CD-HIT or `mmseqs2`. Whole clusters should go into train, validation, or test together.

### G. Use a positional embedding
Right now we're completely disregarding amino acid position in the sequence, and aggregating all information together. By implementing a positional embedding, we retain extra information about the sequence.

## 13. Try out protein structure prediction
In this notebook, we tokenise and embed each amino acid and predict a single value (probability of being soluble). To perform protein structure prediction, we must predict the coordinates of every atom in our protein. This is far too big a task to do in this workshop, but if you want to get an idea of what protein structure prediction looks like in practice, you can try using this online, open-source model: https://esmatlas.com/resources?action=fold

## Final Activity
Return to the **Configuration** section above and try changing the model hyperparameters. See how high you can push the test accuracy.